In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# Setting up the databases

Restricted to the district 12 half of the selected portion of the I-5. Keep only weekdays between 6am and 8pm (adjustable).

## Two DataFrames containing station data
- main for mainline
- connectors for on/off ramps

## Two DataFrames recording nearest neighbor relationships
- station_order for order in which mainline stations appear in each direction
- ramp_neighbors recording, for each main station, the nearest off ramp downstream and nearest on ramp upstream

In [ ]:
# Compile the main station data
data_dir = Path("./Data")

# Select all distrct 12 files
files = sorted(data_dir.rglob("d12_text_station_5min_*.parquet"))

columns = ["timestamp", "station", "speed", "pct_observed", "flow", "direction", "occupancy", "station_length"]

parts = []

for i, path in enumerate(files, start=1):
    # Drop weekends before reading them
    file_date = datetime.strptime(path.stem[-10:], "%Y_%m_%d")
    if file_date.weekday() >=5:
        continue

    # Optional more row filters can go here
    filters_main = [("lane_type", "==", "ML")]
    part = pd.read_parquet(path, columns=columns, engine="pyarrow", filters=filters_main)

    # Restrict TOD; default times were chosen based on average flow drop off
    part = part.loc[
        (part["timestamp"].dt.hour >= 6)
        & (part["timestamp"].dt.hour <= 19)
    ]
    parts.append(part)

    if i % 50 == 0:
        print(f"Loaded {i}/{len(files)} files")

main = pd.concat(parts, ignore_index=True)
del parts

In [ ]:
# Compile on/off ramp data

# Remark: there are 14 stations of type FF (freeway-freeway) on I-5 in district 12
# 7 of them fairly clearly act the same way as on/off ramps, so these are included
# There are 3 more which are HOV specific and tend to get relatively little flow compared to the rest, and they are currently excluded

# The 7 on/off stations labeled as FF act as I-5 northbound entrances/exits
ff_roles = {
    1208990: "exit",      # N5 -> N55
    1209013: "exit",      # N5 -> S55
    1209012: "entrance",  # N55 -> N5
    1210020: "exit",      # N5 -> W22
    1205315: "exit",      # N5 -> N57
    1212015: "entrance",  # E22 -> N5
    1213419: "exit",      # N5 -> W91
}

columns = ["timestamp", "station", "speed", "pct_observed", "flow", "direction", "occupancy", "station_length", "lane_type"]

parts = []

for i, path in enumerate(files, start=1):
    # Drop weekends before reading them
    file_date = datetime.strptime(path.stem[-10:], "%Y_%m_%d")
    if file_date.weekday() >=5:
        continue

    filters_connectors = [("lane_type", "in", ["OR", "FR", "FF"])]
    part = pd.read_parquet(path, columns=columns, engine="pyarrow", filters=filters_connectors)

    # Restrict TOD; default times were chosen based on average flow drop off
    part = part.loc[
        (part["timestamp"].dt.hour >= 6)
        & (part["timestamp"].dt.hour <= 19)
    ]

    # Keep only the mentioned FF stations
    part = part.loc[part["lane_type"].isin(["OR", "FR"]) | part["station"].isin(ff_roles)]
    parts.append(part)

    if i % 50 == 0:
        print(f"Loaded {i}/{len(files)} files")

connectors = pd.concat(parts, ignore_index=True)
del parts

In [ ]:
# Station metadata DataFrame
meta_d12 = pd.read_csv("./Data/d12_text_meta_2021_05_18.txt", sep="\t")

In [ ]:
# Main lane direct neighbor data

# Keep I-5 mainline stations
stations = meta_d12.loc[
    (meta_d12["Fwy"] == 5)
    & (meta_d12["Type"] == "ML")
    & meta_d12["Dir"].isin(["N", "S"]),
    ["ID", "Dir", "Abs_PM", "Name", "Latitude", "Longitude"],
].copy()

stations = stations.loc[stations["ID"].isin(main["station"].unique())]

# Arrange each direction in order
station_order = pd.concat(
    [
        stations.loc[stations["Dir"] == "N"].sort_values("Abs_PM"),
        stations.loc[stations["Dir"] == "S"].sort_values(
            "Abs_PM", ascending=False
        ),
    ],
    ignore_index=True,
)

# Position along the road, counting from 0 within each direction
station_order["road_order"] = station_order.groupby("Dir").cumcount()

# Neighbors relative to the direction of travel
groups = station_order.groupby("Dir")["ID"]
station_order["upstream_id"] = groups.shift(1).astype("Int64")
station_order["downstream_id"] = groups.shift(-1).astype("Int64")

station_order

In [ ]:
# Nearest on/off ramp neighbors to each main lane station

meta = meta_d12.loc[(meta_d12["Fwy"] == 5) & meta_d12["Dir"].isin(["N", "S"])].copy()

meta["Abs_PM"] = pd.to_numeric(meta["Abs_PM"], errors="coerce")
meta = meta.dropna(subset=["Abs_PM"])

# On/off ramp roles
meta["ramp_role"] = meta["Type"].map({
    "OR": "entrance",
    "FR": "exit",
})

is_ff = meta["Type"].eq("FF")
meta.loc[is_ff, "ramp_role"] = meta.loc[is_ff, "ID"].map(ff_roles)

parts = []

for direction in ["N", "S"]:
    road = meta.loc[meta["Dir"] == direction].copy()

    # Set direction of increase of postmiles and sort in that order
    road["travel_pm"] = (road["Abs_PM"] if direction == "N" else -road["Abs_PM"])
    mainline = road.loc[(road["Type"] == "ML") & (road["ID"].isin(main["station"].unique()))].sort_values("travel_pm")

    for _, station in mainline.iterrows():
        row = station[["ID", "Dir", "Abs_PM", "Name"]].to_dict()

        for role, label in [
            ("entrance", "upstream_onramp"),
            ("exit", "downstream_offramp"),
        ]:
            candidates = road.loc[road["ramp_role"] == role]

            if role == "entrance":
                candidates = candidates.loc[candidates["travel_pm"] <= station["travel_pm"]]
                nearest_pm = candidates["travel_pm"].max()
            else:
                candidates = candidates.loc[candidates["travel_pm"] >= station["travel_pm"]]
                nearest_pm = candidates["travel_pm"].min()

            # Retain every detector tied at the nearest postmile.
            nearest = candidates.loc[candidates["travel_pm"] == nearest_pm].sort_values("ID")

            row[f"{label}_distance"] = abs(station["travel_pm"] - nearest_pm)

            for number, ramp_id in enumerate(nearest["ID"], start=1):
                row[f"{label}_{number}_id"] = ramp_id

        parts.append(row)

ramp_neighbors = pd.DataFrame(parts)

# Retain different IDs for ties
id_columns = [column for column in ramp_neighbors.columns if column.endswith("_id")]
ramp_neighbors[id_columns] = ramp_neighbors[id_columns].astype("Int64")

# Features

In [ ]:
model_data = main.copy()

In [ ]:
# Create target feature: speed in 30min
time_col = "timestamp"
station_col = "station"
speed_col = "speed"

model_data[time_col] = pd.to_datetime(model_data[time_col])

future = model_data[[station_col, time_col, speed_col]].copy()
future[time_col] -= pd.Timedelta(minutes=30)
future = future.rename(columns={speed_col: "speed_in_30min"})

model_data = model_data.merge(
    future,
    on=[station_col, time_col],
    how="left",
    validate="one_to_one",
)

model_data["target_time"] = model_data[time_col] + pd.Timedelta(minutes=30)
model_data = model_data.dropna(subset=["speed_in_30min"])

model_data["timestamp"] = pd.to_datetime(model_data["timestamp"])
model_data["target_time"] = pd.to_datetime(model_data["target_time"])

In [ ]:
# Change in speed over last t timestamps at current station

t = 2
interval = pd.Timedelta(minutes=5)

# Use master reference for data
main_speeds = main[["station", "timestamp", "speed"]]
ramp_speeds = connectors[["station", "timestamp", "speed"]]

previous = main_speeds.rename(columns={"speed": "previous_speed"}).copy()
previous["timestamp"] += t*interval

model_data = model_data.merge(previous, on=["station", "timestamp"], how="left", validate="many_to_one")

model_data[f"speed_delta_{t}"] = (model_data["speed"] - model_data["previous_speed"])
model_data = model_data.drop(columns="previous_speed")

In [ ]:
# Downstream and upstream ramp flow

id_columns = [column for column in ramp_neighbors.columns if column.endswith("_id")]

for id_column in id_columns:
    feature_name = id_column.removesuffix("_id") + "_flow"

    lookup = ramp_neighbors[["ID", id_column]].rename(columns={"ID": "station", id_column: "ramp_station"})

    # Attach ramp IDs to the corresponding mainline stations
    matched = model_data[["station", "timestamp"]].merge(
        lookup,
        on="station",
        how="left",
        validate="many_to_one",
    )

    # Add flow on the ramp at the same timestamp
    matched = matched.merge(
        connectors[["station", "timestamp", "flow"]].rename(
            columns={
                "station": "ramp_station",
                "flow": feature_name,
            }
        ),
        on=["ramp_station", "timestamp"],
        how="left",
        validate="many_to_one",
    )

    model_data[feature_name] = matched[feature_name].to_numpy()

In [ ]:
# Downstream and upstream MAINLINE station speeds at current timestamp

for neighbor_col, feature_name in [("upstream_id", "upstream_speed"), ("downstream_id", "downstream_speed")]:
    lookup = station_order[["ID", neighbor_col]].rename(columns={"ID": "station", neighbor_col: "temp_neighbor_station"})

    # Merge neighbors
    model_data = model_data.merge(lookup, on="station", how="left", validate="many_to_one")

    # Match after dropping missing neighbors
    temp_speeds = main_speeds.rename(columns={"station": "temp_neighbor_station", "speed": feature_name}).dropna(subset=["temp_neighbor_station"])
    model_data = model_data.merge(temp_speeds, on=["temp_neighbor_station", "timestamp"], how="left", validate="many_to_one")

    model_data = model_data.drop(columns="temp_neighbor_station")

In [ ]:
# These are almost entirely missing from the data, so probably not useful
# Downstream and upstream RAMP station speeds at current timestamp

ramp_id_col = [column for column in ramp_neighbors.columns if column.startswith(("upstream_onramp_", "downstream_offramp_")) and column.endswith("_id")]

for neighbor_col in ramp_id_col:
    feature_name = neighbor_col.removesuffix("_id") + "_speed"

    lookup = ramp_neighbors[["ID", neighbor_col]].rename(columns={"ID": "station", neighbor_col: "temp_neighbor_station"})
    
    # Merge neighbors
    model_data = model_data.merge(lookup, on="station", how="left", validate="many_to_one")
    
    # Match after dropping missing neighbors
    temp_speeds = ramp_speeds.rename(columns={"station": "temp_neighbor_station", "speed": feature_name}).dropna(subset=["temp_neighbor_station"])
    model_data = model_data.merge(temp_speeds, on=["temp_neighbor_station", "timestamp"], how="left", validate="many_to_one")
    
    model_data = model_data.drop(columns="temp_neighbor_station")

# Train/validate/test split

Using July--December 2025 as final holdout and 4 temporal folds for training

In [ ]:
# The folds are stored as row indices
time = model_data["timestamp"]
target_time = model_data["target_time"]

val_windows = [
    ("2023-07-01", "2024-01-01"),
    ("2024-01-01", "2024-07-01"),
    ("2024-07-01", "2025-01-01"),
    ("2025-01-01", "2025-07-01"),
]

folds = []

for start, end in val_windows:
    # Validation endpoints
    val_start = pd.Timestamp(start)
    val_end = pd.Timestamp(end)

    # Store row indices by flatnonzero-ing boolean values
    folds.append({
        "train_ind": np.flatnonzero(((time >= "2023-01-01") & (target_time < val_start)).to_numpy()),
        "val_ind": np.flatnonzero(((time >= val_start) & (target_time < val_end)).to_numpy()),
    })

# Holdout set row indices
test_start = pd.Timestamp("2025-07-01")
test_end = pd.Timestamp("2026-01-01")

test_ind = np.flatnonzero(((time >= test_start) & (target_time < test_end)).to_numpy())

# Define the split dataframes
for i, fold in enumerate(folds, start=1):
    train = model_data.iloc[fold["train_ind"]]
    validation = model_data.iloc[fold["val_ind"]]
    
test = model_data.iloc[test_ind]